# Spatial Inequities in the Degradation of Night Sky Visibility
### Globe at Night × UNDP HDI — Full Analysis Notebook

**Research question:** Does average naked-eye sky visibility (limiting magnitude) relate to a country's Human Development Index (HDI), and is that relationship linear or nonlinear?

---

### Data Dictionary
| Term | Definition |
|---|---|
| **LimitingMag** | Faintest star visible to the naked eye under current conditions. Higher = darker, less light-polluted sky. Globe at Night scale: 1 (only Moon/planets visible) to ~7 (pristine dark sky). |
| **HDI (Value)** | UNDP Human Development Index — composite of life expectancy, education, and income per capita. Scale: 0–1 (higher = more developed). |
| **valid_data_points** | Observations per country passing quality filtering (LimitingMag in range 1–7). Used to assess reliability of each country's average. |

---

### Notebook Structure
1. Setup & imports
2. Data loading & preprocessing
3. Exploratory visualizations
4. Baseline regression (linear, logarithmic)
5. Nonlinearity investigation
   - 5a. Quadratic regression + F-test
   - 5b. Peak HDI estimation
   - 5c. Sample-size-filtered replication
   - 5d. Piecewise / segmented linear regression
   - 5e. LOWESS on averaged data
   - 5f. HDI tertile split
   - 5g. Summary
6. Advanced models (polynomial CV, Random Forest, Gradient Boosting with Optuna)
7. Model comparison
8. Residual diagnostics
9. Country-level visualizations
10. Final results summary

---
## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from scipy.stats import pearsonr, spearmanr, f as f_dist
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("All imports successful.")

---
## 2. Data Loading & Preprocessing

In [ ]:
# ── 2.1 Load all GaN annual CSV files ──────────────────────────────────────
all_dataframes = []
load_summary   = []

for year in range(2006, 2025):
    f = f"GaN{year}.csv"
    try:
        df = pd.read_csv(f, engine='python')
        df['_source_year'] = year
        all_dataframes.append(df)
        load_summary.append({'Year': year, 'Status': 'Loaded', 'Rows': len(df)})
    except FileNotFoundError:
        load_summary.append({'Year': year, 'Status': 'NOT FOUND', 'Rows': 0})
    except pd.errors.ParserError as e:
        load_summary.append({'Year': year, 'Status': 'ParseError', 'Rows': 0})

load_df = pd.DataFrame(load_summary)
print(load_df.to_string(index=False))
print(f"\nTotal rows loaded: {load_df['Rows'].sum():,}")

if all_dataframes:
    star_data = pd.concat(all_dataframes, axis=0, ignore_index=True)
else:
    raise RuntimeError("No GaN files were loaded. Check that CSV files are in the working directory.")

In [ ]:
# ── 2.2 Clean country names in GaN data ────────────────────────────────────
# Strip region suffixes after hyphens (e.g. "United States - mainland" → "United States")
star_data["Country"] = (
    star_data["Country"]
    .str.split("-", n=1).str[0]
    .str.strip()
)

In [ ]:
# ── 2.3 Load and clean HDI dataset ─────────────────────────────────────────
hdi = pd.read_excel('HDI.xlsx')

# Drop the spurious empty column present in this file
if "mmm nmmmmmnnnnmmmmm" in hdi.columns:
    hdi = hdi.drop(columns=["mmm nmmmmmnnnnmmmmm"])

hdi.columns = hdi.columns.str.strip()

# Strip parenthetical notes and region suffixes from country names
hdi["Country"] = (
    hdi["Country"]
    .str.replace(r"\s*-\s*.*$", "", regex=True)
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.strip()
)

print(f"HDI dataset: {len(hdi)} countries")
hdi.head()

In [ ]:
# ── 2.4 Country name harmonization and territory exclusion ─────────────────
#
# NOTE on ambiguous mappings:
#   - Both "South Korea" and "North Korea" map to "Korea" because the HDI file
#     uses a single "Korea" entry after bracket-stripping. GaN observations
#     for North Korea are extremely rare, so the practical impact is negligible.
#   - "Democratic Republic of the Congo" → "Congo" matches the HDI entry
#     "Congo (Democratic Republic of the)" after bracket-stripping.
#     The Republic of the Congo appears separately as "Congo" in HDI — this
#     ambiguity is treated as a known limitation.

gan_to_hdi = {
    "Turkey":                           "Türkiye",
    "Vietnam":                          "Viet Nam",
    "Russia":                           "Russian Federation",
    "Macedonia":                        "North Macedonia",
    "Czech Republic":                   "Czechia",
    "South Korea":                      "Korea",
    "North Korea":                      "Korea",
    "The Bahamas":                      "Bahamas",
    "Democratic Republic of the Congo": "Congo",
    "Syria":                            "Syrian Arab Republic",
    "Laos":                             "Lao People's Democratic Republic",
    "St Vincent and the Grenadines":    "Saint Vincent and the Grenadines",
    "The Gambia":                       "Gambia",
    "Cape Verde":                       "Cabo Verde",
    "St Kitts and Nevis":               "Saint Kitts and Nevis",
}

# Territories without HDI scores — excluded from analysis
territories_to_exclude = {
    "Puerto Rico", "Cayman Islands", "Taiwan", "Greenland", "Isle of Man",
    "Virgin Islands", "Bermuda", "Guam", "Martinique", "Guernsey", "Aruba",
    "New Caledonia", "Antarctica", "French Polynesia", "Reunion",
    "British Indian Ocean Territory", "British Virgin Islands", "Gibraltar",
    "Cook Islands", "Turks and Caicos Islands", "Norfolk Island", "Baker Island",
    "Pitcairn Islands", "Netherlands Antilles", "Guadeloupe", "Falkland Islands",
    "Wake Island", "South Georgia and the South Sandwich Islands",
    "Northern Mariana Islands",
}

star_data["Country"] = star_data["Country"].replace(gan_to_hdi)
star_data = star_data[~star_data["Country"].isin(territories_to_exclude)].reset_index(drop=True)

In [ ]:
# ── 2.5 Merge HDI into GaN observations ────────────────────────────────────
star_data = star_data.merge(
    hdi[["Country", "Value"]],
    on="Country",
    how="left"
)
star_data["Value"] = pd.to_numeric(star_data["Value"], errors="coerce")

# Report countries in GaN that did not match any HDI entry
unmatched = star_data[star_data["Value"].isna()]["Country"].value_counts().head(20)
print("Top unmatched GaN countries (no HDI entry found):")
print(unmatched.to_string())

In [ ]:
# ── 2.6 Filter to valid LimitingMag values ─────────────────────────────────
#
# The raw data contains physically implausible values (e.g. -4998, 0).
# LimitingMag = 0 is likely a missing-value sentinel, not a real reading.
# The valid range 1–7 is consistent with the Globe at Night magnitude chart;
# values above 7.5 are brighter than the darkest naturally achievable sky.

total_obs = len(star_data)
star_data_valid = star_data[
    (star_data["LimitingMag"] >= 1) &
    (star_data["LimitingMag"] <= 7) &
    (star_data["Value"].notna())
].copy()

dropped = total_obs - len(star_data_valid)
print(f"Total observations before filtering : {total_obs:,}")
print(f"Valid observations after filtering  : {len(star_data_valid):,}")
print(f"Dropped (invalid or unmatched)      : {dropped:,} ({dropped/total_obs*100:.1f}%)")

In [ ]:
# ── 2.7 Build country-averaged dataset ─────────────────────────────────────
total_per_country = star_data.groupby("Country").size().reset_index(name="total_data_points")
valid_per_country = star_data_valid.groupby("Country").size().reset_index(name="valid_data_points")

avg_by_country = (
    star_data_valid
    .groupby("Country")
    .agg(LimitingMag=("LimitingMag", "mean"), Value=("Value", "mean"))
    .reset_index()
)

avg_by_country = (
    avg_by_country
    .merge(total_per_country, on="Country", how="left")
    .merge(valid_per_country, on="Country", how="left")
)
avg_by_country["valid_data_points"] = avg_by_country["valid_data_points"].fillna(0).astype(int)

print(f"Countries with any valid observations: {len(avg_by_country)}")
print("\nTop 10 countries by observation count:")
print(avg_by_country.nlargest(10, "valid_data_points")[["Country", "valid_data_points", "LimitingMag", "Value"]].to_string(index=False))

# plot_data is the primary analysis dataframe used throughout the notebook
plot_data = avg_by_country.dropna(subset=["Value", "LimitingMag"]).copy()
print(f"\nFinal analysis dataset: {len(plot_data)} countries")

In [ ]:
# ── 2.8 Sample-size-filtered dataset ───────────────────────────────────────
# Countries with very few observations have noisy averages. This filtered
# version (≥30 valid observations) is used for sensitivity checks throughout.

MIN_OBS = 30
plot_data_filtered = plot_data[plot_data["valid_data_points"] >= MIN_OBS].copy()
print(f"Countries with >= {MIN_OBS} valid observations: {len(plot_data_filtered)}")
print(f"Countries excluded by threshold     : {len(plot_data) - len(plot_data_filtered)}")

---
## 3. Exploratory Visualizations

In [ ]:
# ── 3.1 Raw scatter: all observations ──────────────────────────────────────
sample = star_data_valid.sample(min(30000, len(star_data_valid)), random_state=42)

plt.figure(figsize=(12, 6))
plt.scatter(sample["Value"], sample["LimitingMag"],
            alpha=0.05, s=5, color="steelblue")
plt.xlabel("HDI Value"); plt.ylabel("Limiting Magnitude")
plt.title("Raw Observations: Limiting Magnitude vs HDI (sample of 30,000 points)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Note: each dot is a single citizen-science observation. The overplotting"
      " reflects the heavy concentration of reports from high-HDI countries.")

In [ ]:
# ── 3.2 LOWESS on raw observations ─────────────────────────────────────────
plt.figure(figsize=(12, 6))
sns.regplot(
    x="Value", y="LimitingMag", data=sample,
    scatter_kws={"alpha": 0.05, "s": 5},
    line_kws={"color": "red", "linewidth": 2},
    lowess=True
)
plt.xlabel("HDI Value"); plt.ylabel("Limiting Magnitude")
plt.title("LOWESS Smoothed Trend — All Raw Observations")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Interpretation: the red LOWESS curve shows the local average at each HDI level."
      " A hump shape (low → high → low) would suggest a nonlinear relationship.")

In [ ]:
# ── 3.3 Country-average scatter (log-scaled bubble size) ───────────────────
fig, ax = plt.subplots(figsize=(12, 7))

sizes = np.log1p(plot_data["valid_data_points"]) * 20
scatter = ax.scatter(
    plot_data["Value"], plot_data["LimitingMag"],
    s=sizes, c=plot_data["LimitingMag"],
    cmap="RdYlGn", alpha=0.75, edgecolors="grey", linewidth=0.4
)
plt.colorbar(scatter, ax=ax, label="Avg Limiting Magnitude")
ax.set_xlabel("HDI Value", fontsize=12)
ax.set_ylabel("Average Limiting Magnitude", fontsize=12)
ax.set_title("Country Averages: Limiting Magnitude vs HDI\n"
             "(bubble area = log(observations) — prevents a few countries from visually dominating)",
             fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Baseline Regression (Linear & Logarithmic)

These monotonic models serve as baselines. A near-zero result here does not rule out a nonlinear relationship — it means the left and right slopes of any hump-shaped curve cancel out.

In [ ]:
# ── 4.1 Pearson & Spearman correlation ─────────────────────────────────────
X_vals = plot_data["Value"].values
y_vals = plot_data["LimitingMag"].values

pearson_r,  pearson_p  = pearsonr(X_vals, y_vals)
spearman_r, spearman_p = spearmanr(X_vals, y_vals)

print("=" * 60)
print("  Linear & Monotonic Correlation: HDI vs Limiting Magnitude")
print("=" * 60)
print(f"  Pearson  r = {pearson_r:+.4f}   p = {pearson_p:.4e}")
print(f"  Spearman r = {spearman_r:+.4f}   p = {spearman_p:.4e}")
sig = lambda p: "Significant (p < 0.05)" if p < 0.05 else "NOT significant"
print(f"  Pearson  → {sig(pearson_p)}")
print(f"  Spearman → {sig(spearman_p)}")
print()
print("Interpretation:")
print("  These tests specifically detect LINEAR and MONOTONIC relationships.")
print("  A near-zero result here does NOT rule out a nonlinear (e.g. inverted-U)")
print("  relationship — it means the positive left half and negative right half")
print("  of any such curve cancel out in both tests.")
print("  Section 5 tests directly for that nonlinear structure.")

In [ ]:
# ── 4.2 Linear regression fit and plot ─────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_1d = X_vals.reshape(-1, 1)

lm = LinearRegression().fit(X_1d, y_vals)
y_lm_pred = lm.predict(X_1d)
r2_lin  = r2_score(y_vals, y_lm_pred)
rmse_lin = np.sqrt(mean_squared_error(y_vals, y_lm_pred))
cv_lin_r2   = cross_val_score(lm, X_1d, y_vals, cv=kf, scoring="r2").mean()
cv_lin_rmse = np.sqrt(-cross_val_score(lm, X_1d, y_vals, cv=kf,
                                        scoring="neg_mean_squared_error").mean())

print(f"Linear Regression — slope: {lm.coef_[0]:+.4f}  intercept: {lm.intercept_:.4f}")
print(f"  Training R²: {r2_lin:.4f}  |  CV R²: {cv_lin_r2:.4f}")
print(f"  Training RMSE: {rmse_lin:.4f}  |  CV RMSE: {cv_lin_rmse:.4f}")
print()
print("Note: a low R² here is expected if the true relationship is nonlinear —")
print("a linear model cannot fit an inverted-U curve.")

x_range = np.linspace(X_vals.min(), X_vals.max(), 300)
plt.figure(figsize=(10, 5))
plt.scatter(X_vals, y_vals, s=np.log1p(plot_data["valid_data_points"])*15,
            alpha=0.6, color="steelblue", edgecolors="grey", linewidth=0.3)
plt.plot(x_range, lm.predict(x_range.reshape(-1,1)), "r-", linewidth=2, label="Linear fit")
plt.xlabel("HDI Value"); plt.ylabel("Avg Limiting Magnitude")
plt.title("Linear Regression: Limiting Magnitude vs HDI (Country Averages)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── 4.3 Logarithmic regression ─────────────────────────────────────────────
plot_data_log = plot_data[plot_data["Value"] > 0].copy()
X_log = np.log(plot_data_log[["Value"]]).values
y_log = plot_data_log["LimitingMag"].values

lm_log = LinearRegression().fit(X_log, y_log)
r2_log  = r2_score(y_log, lm_log.predict(X_log))
rmse_log = np.sqrt(mean_squared_error(y_log, lm_log.predict(X_log)))
cv_log_r2   = cross_val_score(lm_log, X_log, y_log, cv=kf, scoring="r2").mean()
cv_log_rmse = np.sqrt(-cross_val_score(lm_log, X_log, y_log, cv=kf,
                                        scoring="neg_mean_squared_error").mean())

print(f"Logarithmic Regression — R²: {r2_log:.4f}  |  CV R²: {cv_log_r2:.4f}")

---
## 5. Nonlinearity Investigation

The linear correlation tests in Section 4 returned near-zero results, but visual inspection consistently shows a hump-shaped (inverted-U) pattern: sky visibility is lowest for very low-HDI countries, peaks around medium HDI, and declines again at the highest HDI levels.

This section tests whether that nonlinear structure is statistically significant via six independent approaches:
- **5a** — Quadratic regression + F-test for the quadratic term
- **5b** — Estimating the peak HDI value
- **5c** — Replication on sample-size-filtered data (≥30 observations)
- **5d** — Piecewise / segmented linear regression
- **5e** — LOWESS on country-averaged data
- **5f** — HDI tertile split (left vs right slope comparison)
- **5g** — Summary

In [ ]:
# ── 5a. Quadratic regression + F-test ──────────────────────────────────────
#
# We fit two OLS models:
#   Model 1 (linear):    LM = b0 + b1*HDI
#   Model 2 (quadratic): LM = b0 + b1*HDI + b2*HDI²
#
# The F-test compares the two: if the quadratic term (b2) explains a
# statistically significant additional portion of variance, we have
# direct evidence of nonlinearity.

import statsmodels.api as sm

X_lin_sm  = sm.add_constant(X_vals)                          # [1, HDI]
X_quad_sm = sm.add_constant(np.column_stack([X_vals, X_vals**2]))  # [1, HDI, HDI²]

model_linear = sm.OLS(y_vals, X_lin_sm).fit()
model_quad   = sm.OLS(y_vals, X_quad_sm).fit()

print("── QUADRATIC MODEL SUMMARY ──────────────────────────────────────")
print(model_quad.summary())

# F-test: does adding HDI² significantly improve over linear?
# Extra SS = RSS_linear - RSS_quad; df = 1 (one extra parameter)
rss_lin  = model_linear.ssr
rss_quad = model_quad.ssr
n        = len(y_vals)
f_stat   = ((rss_lin - rss_quad) / 1) / (rss_quad / (n - 3))
f_p      = 1 - f_dist.cdf(f_stat, dfn=1, dfd=n-3)

print("\n── F-TEST: Does the quadratic term significantly improve fit? ────")
print(f"   RSS (linear model)    : {rss_lin:.4f}")
print(f"   RSS (quadratic model) : {rss_quad:.4f}")
print(f"   F-statistic           : {f_stat:.4f}")
print(f"   p-value               : {f_p:.4e}")
if f_p < 0.05:
    print("   ✓ The quadratic term is SIGNIFICANT — nonlinear (inverted-U) structure")
    print("     is statistically supported in the data.")
else:
    print("   The quadratic term is not significant at p < 0.05.")
    print("   The inverted-U pattern may be present visually but is not")
    print("   detectable as statistically robust with the current sample size.")

In [ ]:
# ── 5a (continued). Plot quadratic fit against linear fit ──────────────────
b0, b1, b2 = model_quad.params
x_range = np.linspace(X_vals.min(), X_vals.max(), 300)
y_quad_curve = b0 + b1 * x_range + b2 * x_range**2
y_lin_curve  = model_linear.params[0] + model_linear.params[1] * x_range

plt.figure(figsize=(12, 7))
plt.scatter(X_vals, y_vals,
            s=np.log1p(plot_data["valid_data_points"]) * 15,
            alpha=0.65, color="steelblue", edgecolors="grey", linewidth=0.3,
            label="Country average (bubble ∝ log observations)")
plt.plot(x_range, y_lin_curve,  "r--", linewidth=1.8, alpha=0.7, label="Linear fit")
plt.plot(x_range, y_quad_curve, "b-",  linewidth=2.5,
         label=f"Quadratic fit  (R²={model_quad.rsquared:.3f}, p={f_p:.3f})")
plt.xlabel("HDI Value", fontsize=12)
plt.ylabel("Average Limiting Magnitude", fontsize=12)
plt.title("Quadratic vs Linear Fit: Limiting Magnitude vs HDI", fontsize=14)
plt.legend(fontsize=10); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Quadratic model: LM = {b0:.3f} + {b1:+.3f}·HDI + {b2:+.3f}·HDI²")
print(f"Interpretation: b2 = {b2:.3f}. A {'negative' if b2 < 0 else 'positive'} b2 means the curve opens {'downward (inverted-U / peak)' if b2 < 0 else 'upward (U-shape / trough)'}.")

In [ ]:
# ── 5b. Peak HDI — where does sky visibility peak? ─────────────────────────
#
# For a quadratic LM = b0 + b1*x + b2*x², the peak/trough occurs at x = -b1/(2*b2).
# If b2 < 0 (inverted-U), this x value is the HDI at maximum visibility.

if b2 != 0:
    peak_hdi = -b1 / (2 * b2)
    peak_lm  = b0 + b1 * peak_hdi + b2 * peak_hdi**2
    shape = "inverted-U (peak)" if b2 < 0 else "U-shape (trough)"
    print(f"Curve shape: {shape}")
    print(f"Estimated turning point: HDI = {peak_hdi:.3f}  (LimitingMag ≈ {peak_lm:.2f})")

    # Contextualise the peak HDI
    # UNDP thresholds: Low <0.550, Medium 0.550–0.699, High 0.700–0.799, Very High ≥0.800
    if peak_hdi < 0.550:
        tier = "Low HDI"
    elif peak_hdi < 0.700:
        tier = "Medium HDI"
    elif peak_hdi < 0.800:
        tier = "High HDI"
    else:
        tier = "Very High HDI"
    print(f"This falls in the '{tier}' tier (UNDP classification).")

    # Plot the turning point on the fitted curve
    plt.figure(figsize=(10, 6))
    plt.scatter(X_vals, y_vals,
                s=np.log1p(plot_data["valid_data_points"]) * 15,
                alpha=0.6, color="steelblue", edgecolors="grey", linewidth=0.3)
    plt.plot(x_range, y_quad_curve, "b-", linewidth=2.5, label="Quadratic fit")
    plt.axvline(peak_hdi, color="red", linestyle="--", linewidth=1.5,
                label=f"Turning point: HDI = {peak_hdi:.3f}")
    plt.scatter([peak_hdi], [peak_lm], color="red", s=120, zorder=5)
    plt.xlabel("HDI Value"); plt.ylabel("Avg Limiting Magnitude")
    plt.title("Estimated Turning Point of the HDI–Visibility Relationship")
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("b2 = 0: no turning point (linear relationship).")

In [ ]:
# ── 5c. Replication on sample-size-filtered data (≥30 observations) ────────
#
# If the inverted-U pattern is driven by countries with very few observations
# (whose averages are noisy), it should weaken or disappear when we restrict
# to countries with at least 30 valid observations.

X_f = plot_data_filtered["Value"].values
y_f = plot_data_filtered["LimitingMag"].values

X_f_lin  = sm.add_constant(X_f)
X_f_quad = sm.add_constant(np.column_stack([X_f, X_f**2]))

model_f_lin  = sm.OLS(y_f, X_f_lin).fit()
model_f_quad = sm.OLS(y_f, X_f_quad).fit()

rss_f_lin  = model_f_lin.ssr
rss_f_quad = model_f_quad.ssr
n_f        = len(y_f)
f_stat_f   = ((rss_f_lin - rss_f_quad) / 1) / (rss_f_quad / (n_f - 3))
f_p_f      = 1 - f_dist.cdf(f_stat_f, dfn=1, dfd=n_f-3)

b0_f, b1_f, b2_f = model_f_quad.params

print(f"── FILTERED DATA (n={n_f} countries, ≥{MIN_OBS} observations each) ──")
print(f"   Quadratic term coefficient (b2): {b2_f:+.4f}")
print(f"   F-test for quadratic term: F={f_stat_f:.4f}, p={f_p_f:.4e}")
if f_p_f < 0.05:
    print("   ✓ Nonlinear pattern PERSISTS in the filtered dataset.")
    print("   The inverted-U is not an artifact of low-sample-size countries.")
else:
    print("   The nonlinear pattern does not reach significance in the filtered dataset.")
    print("   This suggests small-sample-size countries may be contributing to the shape.")

# Side-by-side: full vs filtered
x_range_f = np.linspace(X_f.min(), X_f.max(), 300)
y_q_full  = b0     + b1     * x_range   + b2     * x_range**2
y_q_filt  = b0_f   + b1_f   * x_range_f + b2_f   * x_range_f**2

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

axes[0].scatter(X_vals, y_vals, s=np.log1p(plot_data["valid_data_points"])*12,
                alpha=0.6, color="steelblue", edgecolors="grey", linewidth=0.3)
axes[0].plot(x_range, y_q_full, "b-", linewidth=2.5,
             label=f"Quadratic (p={f_p:.3f})")
axes[0].set_title(f"All Countries (n={len(plot_data)})", fontweight="bold")
axes[0].set_xlabel("HDI Value"); axes[0].set_ylabel("Avg Limiting Magnitude")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].scatter(X_f, y_f, s=np.log1p(plot_data_filtered["valid_data_points"])*12,
                alpha=0.6, color="darkorange", edgecolors="grey", linewidth=0.3)
axes[1].plot(x_range_f, y_q_filt, "r-", linewidth=2.5,
             label=f"Quadratic (p={f_p_f:.3f})")
axes[1].set_title(f"Filtered: ≥{MIN_OBS} Observations (n={len(plot_data_filtered)})", fontweight="bold")
axes[1].set_xlabel("HDI Value")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Quadratic Fit: Full Dataset vs Sample-Size-Filtered Dataset",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5d. Piecewise (segmented) linear regression ────────────────────────────
#
# Another way to test for the inverted-U: split countries at the estimated
# turning point and fit separate linear regressions to each half.
# If the left slope is positive and the right slope is negative (and both
# are statistically significant), that independently confirms the hump shape.

split_hdi = peak_hdi if b2 != 0 else 0.7  # fallback to 0.7 if no quadratic peak
left_data  = plot_data[plot_data["Value"] <= split_hdi]
right_data = plot_data[plot_data["Value"] >  split_hdi]

def fit_segment(df, label):
    x = df["Value"].values
    y = df["LimitingMag"].values
    X_sm = sm.add_constant(x)
    res  = sm.OLS(y, X_sm).fit()
    slope, slope_p = res.params[1], res.pvalues[1]
    print(f"  {label} (n={len(df)}): slope = {slope:+.4f}  p = {slope_p:.4e}  "
          f"({'Significant' if slope_p < 0.05 else 'Not significant'})")
    return res, x, y

print(f"Split at HDI = {split_hdi:.3f}")
res_left,  x_left,  y_left  = fit_segment(left_data,  "Left segment  (HDI ≤ peak)")
res_right, x_right, y_right = fit_segment(right_data, "Right segment (HDI > peak)")
print()
if res_left.params[1] > 0 and res_right.params[1] < 0:
    print("  Both slopes have the expected direction for an inverted-U:")
    print("  positive left slope (darker skies as HDI rises toward medium)")
    print("  negative right slope (brighter skies as HDI rises to high/very high)")
else:
    print("  Slopes do not both match the inverted-U pattern — see values above.")

# Plot piecewise fit
fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(X_vals, y_vals, s=np.log1p(plot_data["valid_data_points"])*15,
           alpha=0.6, color="steelblue", edgecolors="grey", linewidth=0.3,
           label="Country averages")

x_l = np.linspace(x_left.min(),  split_hdi, 100)
x_r = np.linspace(split_hdi, x_right.max(), 100)
ax.plot(x_l, res_left.params[0]  + res_left.params[1]  * x_l,
        "g-", linewidth=2.5, label=f"Left slope: {res_left.params[1]:+.3f}")
ax.plot(x_r, res_right.params[0] + res_right.params[1] * x_r,
        "r-", linewidth=2.5, label=f"Right slope: {res_right.params[1]:+.3f}")
ax.axvline(split_hdi, color="black", linestyle=":", linewidth=1.5,
           label=f"Split at HDI = {split_hdi:.3f}")
ax.set_xlabel("HDI Value"); ax.set_ylabel("Avg Limiting Magnitude")
ax.set_title("Piecewise Linear Regression: Left and Right of Estimated Peak", fontsize=13)
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ── 5e. LOWESS on country-averaged data ────────────────────────────────────
#
# LOWESS (Locally Weighted Scatterplot Smoothing) fits a flexible local curve
# without assuming any specific functional form. Seeing an inverted-U here
# provides model-free visual confirmation of the nonlinear shape.

from statsmodels.nonparametric.smoothers_lowess import lowess

sorted_idx   = np.argsort(X_vals)
X_sorted     = X_vals[sorted_idx]
y_sorted     = y_vals[sorted_idx]

# Try two bandwidth values to show robustness
lw_50 = lowess(y_sorted, X_sorted, frac=0.50, return_sorted=True)
lw_35 = lowess(y_sorted, X_sorted, frac=0.35, return_sorted=True)

plt.figure(figsize=(12, 7))
plt.scatter(X_vals, y_vals, s=np.log1p(plot_data["valid_data_points"])*15,
            alpha=0.6, color="steelblue", edgecolors="grey", linewidth=0.3,
            label="Country averages")
plt.plot(lw_50[:, 0], lw_50[:, 1], "r-",  linewidth=2.5, label="LOWESS (bandwidth=0.50)")
plt.plot(lw_35[:, 0], lw_35[:, 1], "m--", linewidth=2.0, label="LOWESS (bandwidth=0.35)")
plt.xlabel("HDI Value"); plt.ylabel("Avg Limiting Magnitude")
plt.title("LOWESS on Country Averages — Model-Free Nonlinearity Check", fontsize=13)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Interpretation: if both LOWESS curves show the same hump shape regardless"
      " of bandwidth, the pattern is robust and not an artifact of smoothing.")

In [ ]:
# ── 5f. HDI tertile split — direction test ─────────────────────────────────
#
# Divide countries into three equal-sized HDI groups (tertiles).
# Test: does the middle tertile have significantly higher mean LimitingMag
# than both the low and high tertiles? A significant result (t-tests or ANOVA)
# would confirm the inverted-U structure.

plot_data["HDI_Tertile"] = pd.qcut(
    plot_data["Value"], q=3,
    labels=["Low HDI (T1)", "Medium HDI (T2)", "High HDI (T3)"]
)

tertile_stats = plot_data.groupby("HDI_Tertile", observed=True)["LimitingMag"].agg(
    mean="mean", std="std", n="count", sem=lambda x: x.sem()
).reset_index()
tertile_stats["ci95"] = tertile_stats["sem"] * 1.96

print("Mean Limiting Magnitude by HDI Tertile:")
print(tertile_stats[["HDI_Tertile", "mean", "std", "n"]].to_string(index=False))

# Pairwise t-tests
t1 = plot_data[plot_data["HDI_Tertile"]=="Low HDI (T1)"]["LimitingMag"]
t2 = plot_data[plot_data["HDI_Tertile"]=="Medium HDI (T2)"]["LimitingMag"]
t3 = plot_data[plot_data["HDI_Tertile"]=="High HDI (T3)"]["LimitingMag"]

t12, p12 = stats.ttest_ind(t1, t2)
t23, p23 = stats.ttest_ind(t2, t3)
t13, p13 = stats.ttest_ind(t1, t3)

print("\nPairwise t-tests:")
print(f"  Low vs Medium  : t={t12:+.3f}, p={p12:.4f}  {'*' if p12 < 0.05 else ''}")
print(f"  Medium vs High : t={t23:+.3f}, p={p23:.4f}  {'*' if p23 < 0.05 else ''}")
print(f"  Low vs High    : t={t13:+.3f}, p={p13:.4f}  {'*' if p13 < 0.05 else ''}")
print("  (* = significant at p < 0.05)")

# One-way ANOVA across all three
f_tert, p_tert = stats.f_oneway(t1, t2, t3)
print(f"\nOne-way ANOVA (all three tertiles): F={f_tert:.3f}, p={p_tert:.4f}")

# Bar chart with CI
colors = ["#2196F3", "#4CAF50", "#F44336"]
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(tertile_stats["HDI_Tertile"], tertile_stats["mean"],
              yerr=tertile_stats["ci95"], capsize=8,
              color=colors, edgecolor="black", linewidth=0.8, width=0.5)
for i, row in tertile_stats.iterrows():
    ax.text(i, row["mean"] + row["ci95"] + 0.04, f"{row['mean']:.2f}",
            ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Avg Limiting Magnitude (±95% CI)", fontsize=11)
ax.set_title("Mean Sky Visibility by HDI Tertile\n"
             "An inverted-U pattern would show Middle > Low and Middle > High",
             fontsize=12)
ax.grid(axis="y", alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5g. Nonlinearity summary ────────────────────────────────────────────────
print("=" * 65)
print("  NONLINEARITY INVESTIGATION — SUMMARY")
print("=" * 65)
print()
print(f"  Quadratic term (b2)        : {b2:+.4f} ({'negative = inverted-U' if b2 < 0 else 'positive = U-shape'})")
print(f"  F-test for quadratic term  : F={f_stat:.3f}, p={f_p:.4e}")
print(f"  Estimated turning point    : HDI = {peak_hdi:.3f}")
print()
print(f"  Filtered replication       : F={f_stat_f:.3f}, p={f_p_f:.4e} (n={len(plot_data_filtered)})")
print(f"  Left slope (HDI ≤ peak)    : {res_left.params[1]:+.4f}")
print(f"  Right slope (HDI > peak)   : {res_right.params[1]:+.4f}")
print(f"  Tertile ANOVA              : F={f_tert:.3f}, p={p_tert:.4f}")
print()
print("  Overall conclusion:")
if f_p < 0.05:
    print("  The quadratic term is statistically significant. The consistent")
    print("  inverted-U pattern observed across multiple tests and visualizations")
    print("  is statistically supported — sky visibility peaks at medium HDI")
    print("  and declines for both very low and very high development levels.")
else:
    print("  The quadratic term does not reach statistical significance at p<0.05.")
    print("  However, the consistent direction of the curve across multiple tests")
    print("  (LOWESS, piecewise regression, tertile comparison) suggests a real")
    print("  but statistically underpowered nonlinear trend. The HDI-only model")
    print("  likely has insufficient explanatory power regardless of functional form.")
print("=" * 65)

---
## 6. Advanced Models (Polynomial CV, Random Forest, Gradient Boosting)

In [ ]:
# ── 6.1 Polynomial regression — degree selection via cross-validation ───────
degrees  = range(1, 7)
cv_rmse  = []
cv_r2    = []

for deg in degrees:
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=deg, include_bias=False)),
        ("scaler", StandardScaler()),
        ("reg",    Ridge(alpha=1.0))
    ])
    neg_mse = cross_val_score(pipe, X_1d, y_vals, cv=kf, scoring="neg_mean_squared_error")
    r2_cv   = cross_val_score(pipe, X_1d, y_vals, cv=kf, scoring="r2")
    cv_rmse.append(np.sqrt(-neg_mse.mean()))
    cv_r2.append(r2_cv.mean())

best_deg  = list(degrees)[np.argmin(cv_rmse)]
best_rmse = min(cv_rmse)
best_r2   = cv_r2[np.argmin(cv_rmse)]

print(f"Best polynomial degree (CV): {best_deg}  |  CV RMSE: {best_rmse:.4f}  |  CV R²: {best_r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(list(degrees), cv_rmse, "o-", color="steelblue", linewidth=2)
axes[0].axvline(best_deg, color="red", linestyle="--", label=f"Best degree = {best_deg}")
axes[0].set_xlabel("Polynomial Degree"); axes[0].set_ylabel("CV RMSE")
axes[0].set_title("CV RMSE by Polynomial Degree"); axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].plot(list(degrees), cv_r2, "s-", color="seagreen", linewidth=2)
axes[1].axvline(best_deg, color="red", linestyle="--", label=f"Best degree = {best_deg}")
axes[1].set_xlabel("Polynomial Degree"); axes[1].set_ylabel("CV R²")
axes[1].set_title("CV R² by Polynomial Degree"); axes[1].legend(); axes[1].grid(alpha=0.4)

plt.suptitle("Polynomial Degree Selection via 5-Fold Cross-Validation", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 Fit and plot the best polynomial model ──────────────────────────────
best_pipe = Pipeline([
    ("poly",   PolynomialFeatures(degree=best_deg, include_bias=False)),
    ("scaler", StandardScaler()),
    ("reg",    Ridge(alpha=1.0))
])
best_pipe.fit(X_1d, y_vals)
x_range = np.linspace(X_vals.min(), X_vals.max(), 300).reshape(-1, 1)
y_poly_fit  = best_pipe.predict(x_range)
y_poly_pred = best_pipe.predict(X_1d)
r2_poly   = r2_score(y_vals, y_poly_pred)
rmse_poly = np.sqrt(mean_squared_error(y_vals, y_poly_pred))

plt.figure(figsize=(12, 7))
plt.scatter(X_vals, y_vals, s=np.log1p(plot_data["valid_data_points"])*15,
            c=y_vals, cmap="RdYlGn", alpha=0.8, edgecolors="grey", linewidth=0.4)
plt.plot(x_range, y_poly_fit, "b-", linewidth=2.5,
         label=f"Degree-{best_deg} polynomial (CV R²={best_r2:.3f})")
plt.colorbar(label="Avg Limiting Magnitude")
plt.xlabel("HDI Value"); plt.ylabel("Avg Limiting Magnitude")
plt.title(f"Polynomial (Degree {best_deg}) Regression: Limiting Magnitude vs HDI", fontsize=14)
plt.legend(); plt.grid(alpha=0.35); plt.tight_layout(); plt.show()
print(f"Polynomial (deg {best_deg}) — Training R²: {r2_poly:.4f}  |  RMSE: {rmse_poly:.4f}")

In [ ]:
# ── 6.3 Random Forest — Optuna hyperparameter tuning ───────────────────────
def rf_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 50, 600),
        max_depth         = trial.suggest_int("max_depth", 2, 20),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 15),
        max_features      = trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        random_state      = 42
    )
    return np.sqrt(-cross_val_score(
        RandomForestRegressor(**params), X_1d, y_vals, cv=kf,
        scoring="neg_mean_squared_error").mean())

rf_study = optuna.create_study(direction="minimize")
rf_study.optimize(rf_objective, n_trials=80, show_progress_bar=False)

rf_best = RandomForestRegressor(**rf_study.best_params, random_state=42)
rf_best.fit(X_1d, y_vals)
y_rf_pred  = rf_best.predict(X_1d)
r2_rf      = r2_score(y_vals, y_rf_pred)
rmse_rf    = np.sqrt(mean_squared_error(y_vals, y_rf_pred))
cv_rf_r2   = cross_val_score(rf_best, X_1d, y_vals, cv=kf, scoring="r2").mean()
cv_rf_rmse = np.sqrt(-cross_val_score(rf_best, X_1d, y_vals, cv=kf,
                                       scoring="neg_mean_squared_error").mean())
print(f"Random Forest — CV R²: {cv_rf_r2:.4f}  |  CV RMSE: {cv_rf_rmse:.4f}")

In [ ]:
# ── 6.4 Gradient Boosting — Optuna hyperparameter tuning ───────────────────
def gb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 50, 500),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth         = trial.suggest_int("max_depth", 2, 8),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 15),
        subsample         = trial.suggest_float("subsample", 0.5, 1.0),
        random_state      = 42
    )
    return np.sqrt(-cross_val_score(
        GradientBoostingRegressor(**params), X_1d, y_vals, cv=kf,
        scoring="neg_mean_squared_error").mean())

gb_study = optuna.create_study(direction="minimize")
gb_study.optimize(gb_objective, n_trials=80, show_progress_bar=False)

gb_best = GradientBoostingRegressor(**gb_study.best_params, random_state=42)
gb_best.fit(X_1d, y_vals)
y_gb_pred  = gb_best.predict(X_1d)
r2_gb      = r2_score(y_vals, y_gb_pred)
rmse_gb    = np.sqrt(mean_squared_error(y_vals, y_gb_pred))
cv_gb_r2   = cross_val_score(gb_best, X_1d, y_vals, cv=kf, scoring="r2").mean()
cv_gb_rmse = np.sqrt(-cross_val_score(gb_best, X_1d, y_vals, cv=kf,
                                       scoring="neg_mean_squared_error").mean())
print(f"Gradient Boosting — CV R²: {cv_gb_r2:.4f}  |  CV RMSE: {cv_gb_rmse:.4f}")

---
## 7. Model Comparison

In [ ]:
# ── 7. All models side by side ──────────────────────────────────────────────
results = pd.DataFrame({
    "Model":      ["Linear", "Logarithmic",
                   "Quadratic (OLS)",
                   f"Polynomial (deg {best_deg}, Ridge)",
                   "Random Forest (Optuna)",
                   "Gradient Boosting (Optuna)"],
    "Train R²":   [r2_lin, r2_log,
                   model_quad.rsquared,
                   r2_poly, r2_rf, r2_gb],
    "CV R²":      [cv_lin_r2, cv_log_r2,
                   None,          # OLS quadratic CV R² computed below
                   best_r2, cv_rf_r2, cv_gb_r2],
    "Train RMSE": [rmse_lin, rmse_log,
                   np.sqrt(model_quad.mse_resid),
                   rmse_poly, rmse_rf, rmse_gb],
    "CV RMSE":    [cv_lin_rmse, cv_log_rmse,
                   None,
                   best_rmse, cv_rf_rmse, cv_gb_rmse],
})

# Fill in CV metrics for OLS quadratic
quad_pipe = Pipeline([
    ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
    ("reg",    LinearRegression())
])
cv_quad_r2   = cross_val_score(quad_pipe, X_1d, y_vals, cv=kf, scoring="r2").mean()
cv_quad_rmse = np.sqrt(-cross_val_score(quad_pipe, X_1d, y_vals, cv=kf,
                                         scoring="neg_mean_squared_error").mean())
results.loc[results["Model"]=="Quadratic (OLS)", "CV R²"]   = cv_quad_r2
results.loc[results["Model"]=="Quadratic (OLS)", "CV RMSE"] = cv_quad_rmse

results = results.sort_values("CV R²", ascending=False).reset_index(drop=True)

print("=" * 80)
print("  MODEL COMPARISON — HDI vs Limiting Magnitude (Country Averages)")
print("=" * 80)
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}" if x is not None else "N/A"))
print("=" * 80)
print()
print("Note: low R² values across all models indicate that HDI alone is not a")
print("strong predictor of limiting magnitude. This is itself an important result.")
print("The quadratic model's CV improvement over linear quantifies how much of the")
print("signal is nonlinear rather than simply absent.")

# Visualization — allow negative R² values on x-axis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#CCB974"]

cv_r2_vals = results["CV R²"].astype(float)
cv_rmse_vals = results["CV RMSE"].astype(float)

axes[0].barh(results["Model"], cv_r2_vals, color=colors)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Cross-Validated R²")
axes[0].set_title("CV R² by Model (higher = better)")
axes[0].grid(axis="x", alpha=0.4)
for i, v in enumerate(cv_r2_vals):
    axes[0].text(v + 0.002, i, f"{v:.3f}", va="center")

axes[1].barh(results["Model"], cv_rmse_vals, color=colors)
axes[1].set_xlabel("Cross-Validated RMSE")
axes[1].set_title("CV RMSE by Model (lower = better)")
axes[1].grid(axis="x", alpha=0.4)
for i, v in enumerate(cv_rmse_vals):
    axes[1].text(v + 0.005, i, f"{v:.3f}", va="center")

plt.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

---
## 8. Residual Diagnostics

In [ ]:
# ── 8. Residual diagnostics for the quadratic OLS model ────────────────────
# We use the quadratic model here because it is the primary model of interest
# given the nonlinearity findings in Section 5.

y_quad_pred = model_quad.fittedvalues
residuals   = y_vals - y_quad_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_quad_pred, residuals, alpha=0.6, edgecolors="k", linewidth=0.4)
axes[0].axhline(0, color="red", linewidth=1.5)
axes[0].set_xlabel("Fitted Values"); axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted"); axes[0].grid(alpha=0.35)

axes[1].hist(residuals, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution"); axes[1].grid(alpha=0.35)

stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot (Normality Check)"); axes[2].grid(alpha=0.35)

plt.suptitle("Residual Diagnostics — Quadratic OLS Model", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

sw_stat, sw_p = stats.shapiro(residuals)
print(f"Shapiro-Wilk normality test: W={sw_stat:.4f}, p={sw_p:.4f}")
print("Residuals appear normally distributed." if sw_p > 0.05
      else "Residuals deviate from normality (p < 0.05).")
print()
print("Interpretation: structured residuals (e.g. a remaining curve in the")
print("residuals-vs-fitted plot) would suggest additional predictors are needed.")

---
## 9. Country-Level Visualizations

In [ ]:
# ── 9.1 Top/bottom 20 countries — filtered to ≥30 observations ─────────────
# Rankings are restricted to countries with at least 30 valid observations
# to avoid rankings dominated by countries with very small, noisy samples.

country_plot = plot_data_filtered[["Country", "LimitingMag", "Value", "valid_data_points"]].copy()
country_plot = country_plot.sort_values("LimitingMag", ascending=False)

top20    = country_plot.head(20)
bottom20 = country_plot.tail(20).sort_values("LimitingMag")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(top20["Country"], top20["LimitingMag"],
             color=plt.cm.YlGn(np.linspace(0.4, 0.9, len(top20))))
axes[0].set_xlabel("Avg Limiting Magnitude (higher = darker sky)")
axes[0].set_title(f"Top 20 Darkest Sky Countries\n(≥{MIN_OBS} valid observations)", fontweight="bold")
axes[0].grid(axis="x", alpha=0.35)
for i, (mag, hdi_v, n) in enumerate(zip(top20["LimitingMag"], top20["Value"], top20["valid_data_points"])):
    axes[0].text(mag + 0.03, i, f"{mag:.2f}  HDI={hdi_v:.3f}  n={n}", va="center", fontsize=7.5)

axes[1].barh(bottom20["Country"], bottom20["LimitingMag"],
             color=plt.cm.OrRd(np.linspace(0.4, 0.9, len(bottom20))))
axes[1].set_xlabel("Avg Limiting Magnitude (lower = brighter / more polluted)")
axes[1].set_title(f"Top 20 Most Light-Polluted Countries\n(≥{MIN_OBS} valid observations)", fontweight="bold")
axes[1].grid(axis="x", alpha=0.35)
for i, (mag, hdi_v, n) in enumerate(zip(bottom20["LimitingMag"], bottom20["Value"], bottom20["valid_data_points"])):
    axes[1].text(mag + 0.03, i, f"{mag:.2f}  HDI={hdi_v:.3f}  n={n}", va="center", fontsize=7.5)

plt.suptitle("Country Rankings by Average Limiting Magnitude (sample-size filtered)",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.2 HDI quartile box plot + bar chart ──────────────────────────────────
country_plot["HDI_Quartile"] = pd.qcut(
    country_plot["Value"], q=4,
    labels=["Low (Q1)", "Med-Low (Q2)", "Med-High (Q3)", "High (Q4)"]
)

quartile_stats = country_plot.groupby("HDI_Quartile", observed=True)["LimitingMag"].agg(
    mean="mean", sem=lambda x: x.sem()
).reset_index()
quartile_stats["ci95"] = quartile_stats["sem"] * 1.96

palette = {"Low (Q1)": "#2196F3", "Med-Low (Q2)": "#4CAF50",
           "Med-High (Q3)": "#FF9800", "High (Q4)": "#F44336"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.boxplot(x="HDI_Quartile", y="LimitingMag", data=country_plot,
            palette=palette, ax=axes[0])
axes[0].set_title("Limiting Magnitude by HDI Quartile", fontweight="bold")
axes[0].set_xlabel("HDI Quartile"); axes[0].set_ylabel("Avg Limiting Magnitude")
axes[0].grid(axis="y", alpha=0.35)

axes[1].bar(quartile_stats["HDI_Quartile"], quartile_stats["mean"],
            yerr=quartile_stats["ci95"], capsize=6,
            color=list(palette.values()), edgecolor="black", linewidth=0.8)
axes[1].set_title("Mean Limiting Magnitude by HDI Quartile (±95% CI)", fontweight="bold")
axes[1].set_xlabel("HDI Quartile"); axes[1].set_ylabel("Mean Avg Limiting Magnitude")
axes[1].grid(axis="y", alpha=0.35)
for i, row in quartile_stats.iterrows():
    axes[1].text(i, row["mean"] + row["ci95"] + 0.03, f"{row['mean']:.2f}",
                 ha="center", fontsize=9)

plt.tight_layout(); plt.show()

groups = [g["LimitingMag"].values for _, g in country_plot.groupby("HDI_Quartile", observed=True)]
f_anova, p_anova = stats.f_oneway(*groups)
print(f"One-way ANOVA across HDI quartiles: F={f_anova:.3f}, p={p_anova:.4f}")

In [ ]:
# ── 9.3 Final annotated scatter with quadratic fit overlaid ─────────────────
# Bug fix from previous version: the overlaid curve now correctly uses the
# quadratic OLS model (the primary nonlinearity model) rather than the RF model.

fig, ax = plt.subplots(figsize=(13, 8))
scatter = ax.scatter(
    plot_data["Value"], plot_data["LimitingMag"],
    c=plot_data["LimitingMag"], cmap="RdYlGn",
    s=np.log1p(plot_data["valid_data_points"]) * 20,
    alpha=0.75, edgecolors="grey", linewidth=0.5
)
plt.colorbar(scatter, ax=ax, label="Avg Limiting Magnitude")

# Overlay quadratic OLS fit
x_line = np.linspace(plot_data["Value"].min(), plot_data["Value"].max(), 300)
y_line = b0 + b1 * x_line + b2 * x_line**2
ax.plot(x_line, y_line, "b--", linewidth=2,
        label=f"Quadratic OLS fit (R²={model_quad.rsquared:.3f}, F-test p={f_p:.3f})")

# Label the top/bottom 10% outliers
thr_top    = plot_data["LimitingMag"].quantile(0.90)
thr_bottom = plot_data["LimitingMag"].quantile(0.10)
outliers   = plot_data[(plot_data["LimitingMag"] >= thr_top) | (plot_data["LimitingMag"] <= thr_bottom)]
for _, row in outliers.iterrows():
    ax.annotate(row["Country"], xy=(row["Value"], row["LimitingMag"]),
                xytext=(5, 3), textcoords="offset points", fontsize=7.5, alpha=0.85)

ax.set_xlabel("HDI Value", fontsize=12)
ax.set_ylabel("Average Limiting Magnitude", fontsize=12)
ax.set_title("Light Pollution vs Human Development Index — All Countries\n"
             "(bubble size = log(observations); labelled = top/bottom 10%)",
             fontsize=13, fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 10. Final Results Summary

In [ ]:
# ── 10. Final results summary ───────────────────────────────────────────────
best_row = results.iloc[0]

print("=" * 70)
print("  FINAL RESEARCH RESULTS SUMMARY")
print("  Spatial Inequities in Night Sky Visibility")
print("  Globe at Night (2006–2024) × UNDP HDI")
print("=" * 70)

print(f"\n  Dataset")
print(f"    Total valid observations          : {len(star_data_valid):,}")
print(f"    Countries (all)                   : {len(plot_data)}")
print(f"    Countries (>={MIN_OBS} observations) : {len(plot_data_filtered)}")
print(f"    HDI range                         : {plot_data['Value'].min():.3f} – {plot_data['Value'].max():.3f}")
print(f"    Limiting Mag range                : {plot_data['LimitingMag'].min():.2f} – {plot_data['LimitingMag'].max():.2f}")

print(f"\n  Linear Correlation Tests (cannot detect nonlinearity)")
print(f"    Pearson  r = {pearson_r:+.4f}  p = {pearson_p:.4e}  {'(not significant)' if pearson_p >= 0.05 else '(significant)'}")
print(f"    Spearman r = {spearman_r:+.4f}  p = {spearman_p:.4e}  {'(not significant)' if spearman_p >= 0.05 else '(significant)'}")
print(f"    Interpretation: near-zero monotonic correlation is expected if")
print(f"    the true relationship is nonlinear (positive and negative halves cancel).")

print(f"\n  Nonlinearity Tests")
print(f"    Quadratic term (b2) = {b2:+.4f} ({'inverted-U' if b2 < 0 else 'U-shape'})")
print(f"    F-test (quadratic vs linear): F={f_stat:.3f}, p={f_p:.4e}")
print(f"    Estimated turning point: HDI ≈ {peak_hdi:.3f}")
print(f"    Filtered replication (n={len(plot_data_filtered)}): F={f_stat_f:.3f}, p={f_p_f:.4e}")
print(f"    Tertile ANOVA: F={f_tert:.3f}, p={p_tert:.4f}")

print(f"\n  Best Model: {best_row['Model']}")
print(f"    CV R²: {float(best_row['CV R²']):.4f}  |  CV RMSE: {float(best_row['CV RMSE']):.4f}")
print(f"    Note: low R² confirms HDI alone is a weak predictor. The nonlinear")
print(f"    structure is real but explains only a fraction of the total variance.")

print(f"\n  Conclusion")
if f_p < 0.05:
    print(f"    A statistically significant nonlinear (inverted-U) relationship")
    print(f"    exists between HDI and average limiting magnitude. Sky visibility")
    print(f"    peaks around HDI ≈ {peak_hdi:.2f} and declines for both lower and")
    print(f"    higher development levels. Linear correlation tests returned")
    print(f"    non-significant results because they cannot detect this shape.")
else:
    print(f"    A consistent inverted-U trend appears across all visual and")
    print(f"    model-based tests but does not reach p<0.05 significance.")
    print(f"    HDI alone explains little variance in limiting magnitude;")
    print(f"    additional predictors (population density, energy consumption,")
    print(f"    urbanization rate) are likely needed to build a robust model.")
print("=" * 70)